# Experiment 1: Few-shot MT

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# ── INSTALL DEPENDENCIES ─────────────────────────────────────────
!pip install -q transformers accelerate datasets sacrebleu evaluate \
    sentencepiece protobuf bitsandbytes peft trl

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — switch runtime!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# ── GLOBAL CONFIG ────────────────────────────────────────────────

LANG_PAIRS = [
    ('eng_Latn', 'fra_Latn', 'English', 'French'),   # High resource
    ('fra_Latn', 'eng_Latn', 'French', 'English'),   # Reverse
    ('eng_Latn', 'xho_Latn', 'English', 'Xhosa'),    # Low resource
]

import json, re, random, pandas as pd, numpy as np
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import evaluate
from tqdm.notebook import tqdm

random.seed(42)
np.random.seed(42)

In [ ]:
# ── UTILITY FUNCTIONS (batched / parallelized generation) ─────────────

def load_flores(src_lang, tgt_lang, split='devtest', n=50):
    src_ds = load_dataset('openlanguagedata/flores_plus', src_lang, split=split)
    tgt_ds = load_dataset('openlanguagedata/flores_plus', tgt_lang, split=split)
    return [ex['text'] for ex in src_ds][:n], [ex['text'] for ex in tgt_ds][:n]

# Quick test
test_src, test_ref = load_flores('eng_Latn', 'fra_Latn', split='devtest', n=3)
for s, r in zip(test_src, test_ref):
    print(f"EN: {s}\nFR: {r}\n")


def make_translation_prompt(
    src_text: str,
    src_name: str,
    tgt_name: str,
    model_family: str = "generic",
    thinking: bool = True,
) -> str:
    prompt = (
        f"Please write a high-quality {tgt_name} translation of the following {src_name} sentence\n"
        f"\"{src_text}\"\n"
        "Please provide only the translation, nothing more.\n"
    )

    # Paper's Qwen non-thinking mode
    if model_family.lower().startswith("qwen") and not thinking:
        prompt += "<think>\n\n</think>\n"

    return prompt


import re

def _extract_translation(decoded_text: str) -> str:
    if not decoded_text:
        return ""

    text = decoded_text.strip()

    # Remove thinking traces
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

    # Prefer content after "Final Translation" if present
    m = re.search(r"Final Translation\s*:?\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    # Drop any remaining tags
    text = re.sub(r"</?[^>]+>", "", text).strip()

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text.strip(" \"'")


def _build_chat_prompts(sources, prompt_fn, enable_thinking=False):
    prompts = []
    for src in sources:
        msgs = [{'role': 'user', 'content': prompt_fn(src)}]
        prompts.append(
            tokenizer.apply_chat_template(
                msgs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking,
            )
        )
    return prompts


def translate(
    model,
    tokenizer,
    sources,
    prompt_fn,
    max_new_tokens=256,
    enable_thinking=False,
    batch_size=1024,
):
    """
    Batched generation for much better GPU utilization on Colab.
    Falls back to a smaller batch automatically if CUDA OOM happens.
    """
    prompts = _build_chat_prompts(sources, prompt_fn, enable_thinking=enable_thinking)
    translations = []
    i = 0

    pbar = tqdm(total=len(prompts))
    while i < len(prompts):
        current_bs = min(batch_size, len(prompts) - i)

        while current_bs >= 1:
            batch_prompts = prompts[i:i + current_bs]
            try:
                inputs = tokenizer(
                    batch_prompts,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=None,
                        top_p=None,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                        use_cache=True,
                    )

                # IMPORTANT: generated tokens start after the padded input width,
                # not after each sample's non-pad length.
                prompt_width = inputs['input_ids'].shape[1]

                for row in outputs:
                    generated_tokens = row[prompt_width:]
                    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                    translations.append(_extract_translation(decoded))

                i += current_bs
                pbar.update(current_bs)
                break

            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                current_bs //= 2
                if current_bs == 0:
                    raise RuntimeError(
                        'Out of memory even with batch_size=1. '
                        'Lower max_new_tokens or use a smaller model.'
                    )

    pbar.close()
    return translations


def score(hyps, refs):
    bleu = evaluate.load('sacrebleu')
    chrf = evaluate.load('chrf')
    b = bleu.compute(predictions=hyps, references=[[r] for r in refs])['score']
    c = chrf.compute(predictions=hyps, references=[[r] for r in refs], word_order=2)['score']
    return round(b, 2), round(c, 2)

print('Utilities loaded! Batched translation is enabled.')

In [ ]:
# ── EXP 1: FEW-SHOT MT WITH GEMMA-3-1B ─────────────

# -----------------------------
# Config
# -----------------------------
GEMMA_MODEL_ID = "google/gemma-3-1b-it"
ICL_K_VALUES = [0, 1, 2, 4, 8]   # nested prefixes: fair comparison across k
N_EVAL_EXP1 = 50
SEED_EXP1 = 13
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# We need at least max(k) support examples plus evaluation examples.
K_MAX = max(ICL_K_VALUES)
N_TOTAL_NEEDED = K_MAX + N_EVAL_EXP1

random.seed(SEED_EXP1)
np.random.seed(SEED_EXP1)
torch.manual_seed(SEED_EXP1)

# -----------------------------
# Load Gemma-1B IT
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()

# -----------------------------
# Helpers
# -----------------------------
def make_fewshot_translation_prompt(src_text, src_name, tgt_name, demonstrations):
    """
    Few-shot prompt for MT.
    demonstrations: list of (src_sent, tgt_sent)
    """
    lines = [
        f"Translate from {src_name} to {tgt_name}.",
        "Use the examples below to infer the translation pattern.",
        "Return only the translation, nothing else.",
        "",
    ]

    if demonstrations:
        lines.append("Examples:")
        for i, (ex_src, ex_tgt) in enumerate(demonstrations, start=1):
            lines.extend([
                f"Example {i}",
                f"{src_name}: {ex_src}",
                f"{tgt_name}: {ex_tgt}",
                ""
            ])

    lines.extend([
        "Now translate:",
        f"{src_name}: {src_text}",
        f"{tgt_name}:"
    ])
    return "\n".join(lines)


def get_fair_support_and_eval(src_lang, tgt_lang, n_eval, k_max):
    """
    Fair setup:
    - fetch exactly k_max + n_eval aligned examples
    - use the FIRST k_max as the fixed support pool
    - use the REMAINDER as the eval set
    This guarantees:
      * same eval set for all k
      * same support pool for all k
      * k-shot prompts use nested prefixes of the same pool
    """
    sources_all, references_all = load_flores(src_lang, tgt_lang, n=k_max + n_eval)

    support_sources = sources_all[:k_max]
    support_refs = references_all[:k_max]

    eval_sources = sources_all[k_max:k_max + n_eval]
    eval_refs = references_all[k_max:k_max + n_eval]

    support_pairs = list(zip(support_sources, support_refs))
    return support_pairs, eval_sources, eval_refs


def summarize_prompt_length_chars(prompt_fn, sources, n_probe=5):
    probe = sources[:min(n_probe, len(sources))]
    if not probe:
        return 0.0
    return float(np.mean([len(prompt_fn(x)) for x in probe]))


# -----------------------------
# Run experiment
# -----------------------------
exp1_results = []
exp1_support_sets = []

for src_lang, tgt_lang, src_name, tgt_name in LANG_PAIRS:
    print(f"\n{'='*100}")
    print(f"GEMMA FEW-SHOT ICL — {src_name} → {tgt_name}")
    print(f"{'='*100}")

    support_pairs, eval_sources, eval_refs = get_fair_support_and_eval(
        src_lang=src_lang,
        tgt_lang=tgt_lang,
        n_eval=N_EVAL_EXP1,
        k_max=K_MAX,
    )

    # Save the fixed support pool used for this direction
    exp1_support_sets.append({
        "src": src_name,
        "tgt": tgt_name,
        "k_max": K_MAX,
        "support_examples": [
            {
                "example_id": i,
                "source": s,
                "target": t,
            }
            for i, (s, t) in enumerate(support_pairs)
        ]
    })

    # Show the support pool once so fairness is explicit
    print(f"Fixed support pool size: {len(support_pairs)}")
    for i, (s, t) in enumerate(support_pairs[:min(2, len(support_pairs))], start=1):
        print(f"\nSupport example {i}")
        print(f"{src_name}: {s}")
        print(f"{tgt_name}: {t}")

    for k in ICL_K_VALUES:
        demos_k = support_pairs[:k]   # nested prefix => fair comparison

        fn = lambda s, sn=src_name, tn=tgt_name, demos=demos_k: make_fewshot_translation_prompt(
            s, sn, tn, demos
        )

        avg_prompt_chars = summarize_prompt_length_chars(fn, eval_sources, n_probe=5)

        hyps = translate(
            model,
            tokenizer,
            eval_sources,
            fn,
            max_new_tokens=160,
            enable_thinking=False,
            batch_size=50,
        )

        bleu, chrf = score(hyps, eval_refs)

        print(
            f"k={k:2d}  "
            f"BLEU: {bleu:6.2f}  "
            f"chrF++: {chrf:6.2f}  "
            f"avg prompt chars: {avg_prompt_chars:7.1f}"
        )

        exp1_results.append({
            "src": src_name,
            "tgt": tgt_name,
            "k": k,
            "bleu": bleu,
            "chrf": chrf,
            "n_eval": len(eval_sources),
            "avg_prompt_chars": avg_prompt_chars,
            "translations": hyps[:10],   # qualitative inspection
        })

# -----------------------------
# Summaries and saving
# -----------------------------
df1 = pd.DataFrame([
    {k: v for k, v in row.items() if k != "translations"}
    for row in exp1_results
])

print(f"\n{'='*100}")
print("EXP 1 SUMMARY")
print(f"{'='*100}")
print(df1.to_string(index=False))

# Mean over directions for each k
df1_mean = (
    df1.groupby("k", as_index=False)
       .agg({
           "bleu": "mean",
           "chrf": "mean",
           "avg_prompt_chars": "mean",
       })
       .sort_values("k")
)

print(f"\n{'='*100}")
print("MEAN ACROSS DIRECTIONS")
print(f"{'='*100}")
print(df1_mean.to_string(index=False))

with open("results_exp1_gemma_fewshot.json", "w", encoding="utf-8") as f:
    json.dump(exp1_results, f, indent=2, ensure_ascii=False)

with open("results_exp1_gemma_support_sets.json", "w", encoding="utf-8") as f:
    json.dump(exp1_support_sets, f, indent=2, ensure_ascii=False)

df1.to_csv("results_exp1_gemma_fewshot.csv", index=False, encoding="utf-8")
df1_mean.to_csv("results_exp1_gemma_fewshot_mean_by_k.csv", index=False, encoding="utf-8")